# 22 — Pipeline Wrapper Pattern

Purpose: wrap a Spark pipeline into one reusable function.

Process schema:

```text
INPUT DATAFRAME
      |
      v
run_pipeline(df, config)
      |
      +--> silver
      +--> quarantine
      +--> metrics
```

Main idea:

```text
One function owns the pipeline flow.
It returns named outputs.
```

In [1]:
from pyspark.sql import SparkSession, DataFrame, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("pipeline-wrapper-pattern")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_patterns_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Config

In [2]:
config = {
    "allowed_event_types": ["purchase", "refund"],
    "min_amount": 0.0,
    "pipeline_name": "customer-events",
    "pipeline_run_id": "run_001",
}

config

{'allowed_event_types': ['purchase', 'refund'],
 'min_amount': 0.0,
 'pipeline_name': 'customer-events',
 'pipeline_run_id': 'run_001'}

## Step 2 — Input data

In [3]:
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time_raw", StringType(), True),
])

rows = [
    ("e1", "u1", "Purchase", 10.0, "2026-01-01 10:00:00"),
    ("e2", None, "purchase", 20.0, "2026-01-01 10:05:00"),
    ("e3", "u2", "refund", 5.0, "2026-01-01 10:10:00"),
    ("e4", "u3", "unknown", 7.0, "2026-01-01 10:15:00"),
    ("e5", "u4", "purchase", -1.0, "2026-01-01 10:20:00"),
    ("e6", "u5", "refund", None, "bad-date"),
]

bronze = spark.createDataFrame(rows, schema)

bronze.show(truncate=False)

+--------+-------+----------+------+-------------------+
|event_id|user_id|event_type|amount|event_time_raw     |
+--------+-------+----------+------+-------------------+
|e1      |u1     |Purchase  |10.0  |2026-01-01 10:00:00|
|e2      |NULL   |purchase  |20.0  |2026-01-01 10:05:00|
|e3      |u2     |refund    |5.0   |2026-01-01 10:10:00|
|e4      |u3     |unknown   |7.0   |2026-01-01 10:15:00|
|e5      |u4     |purchase  |-1.0  |2026-01-01 10:20:00|
|e6      |u5     |refund    |NULL  |bad-date           |
+--------+-------+----------+------+-------------------+



## Step 3 — Small pipeline functions

In [4]:
def normalize(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("event_type", F.lower(F.trim(F.col("event_type"))))
        .withColumn("user_id", F.trim(F.col("user_id")))
    )


def parse_event_time(df: DataFrame) -> DataFrame:
    return df.withColumn("event_time", F.to_timestamp("event_time_raw"))


def apply_rules(df: DataFrame, cfg: dict) -> DataFrame:
    return (
        df
        .withColumn(
            "error_reason",
            F.when(F.col("event_id").isNull(), F.lit("missing_event_id"))
             .when(F.col("user_id").isNull() | (F.col("user_id") == ""), F.lit("missing_user_id"))
             .when(~F.col("event_type").isin(cfg["allowed_event_types"]), F.lit("invalid_event_type"))
             .when(F.col("amount").isNull(), F.lit("missing_amount"))
             .when(F.col("amount") < F.lit(cfg["min_amount"]), F.lit("invalid_amount"))
             .when(F.col("event_time").isNull(), F.lit("invalid_event_time"))
        )
        .withColumn("is_valid", F.col("error_reason").isNull())
    )


def add_audit_columns(df: DataFrame, cfg: dict) -> DataFrame:
    return (
        df
        .withColumn("pipeline_name", F.lit(cfg["pipeline_name"]))
        .withColumn("pipeline_run_id", F.lit(cfg["pipeline_run_id"]))
        .withColumn("processed_at", F.current_timestamp())
    )

## Step 4 — Pipeline wrapper

In [5]:
def run_pipeline(df: DataFrame, cfg: dict) -> dict:
    prepared = (
        df
        .transform(normalize)
        .transform(parse_event_time)
        .transform(lambda d: apply_rules(d, cfg))
        .transform(lambda d: add_audit_columns(d, cfg))
    )

    silver = (
        prepared
        .filter(F.col("is_valid"))
        .select(
            "event_id",
            "user_id",
            "event_type",
            "amount",
            "event_time",
            "pipeline_name",
            "pipeline_run_id",
            "processed_at",
        )
    )

    quarantine = (
        prepared
        .filter(~F.col("is_valid"))
        .select(
            "event_id",
            "user_id",
            "event_type",
            "amount",
            "event_time_raw",
            "error_reason",
            "pipeline_name",
            "pipeline_run_id",
            "processed_at",
        )
    )

    metrics_schema = StructType([
        StructField("metric", StringType(), False),
        StructField("value", LongType(), False),
    ])

    metrics = spark.createDataFrame(
        [
            ("input_rows", df.count()),
            ("prepared_rows", prepared.count()),
            ("silver_rows", silver.count()),
            ("quarantine_rows", quarantine.count()),
        ],
        metrics_schema,
    )

    return {
        "prepared": prepared,
        "silver": silver,
        "quarantine": quarantine,
        "metrics": metrics,
    }

## Step 5 — Run pipeline

In [6]:
result = run_pipeline(bronze, config)

result["silver"].show(truncate=False)
result["quarantine"].show(truncate=False)
result["metrics"].show(truncate=False)

+--------+-------+----------+------+-------------------+---------------+---------------+-------------------------+
|event_id|user_id|event_type|amount|event_time         |pipeline_name  |pipeline_run_id|processed_at             |
+--------+-------+----------+------+-------------------+---------------+---------------+-------------------------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|customer-events|run_001        |2026-05-02 15:51:53.30504|
|e3      |u2     |refund    |5.0   |2026-01-01 10:10:00|customer-events|run_001        |2026-05-02 15:51:53.30504|
+--------+-------+----------+------+-------------------+---------------+---------------+-------------------------+

+--------+-------+----------+------+-------------------+------------------+---------------+---------------+--------------------------+
|event_id|user_id|event_type|amount|event_time_raw     |error_reason      |pipeline_name  |pipeline_run_id|processed_at              |
+--------+-------+----------+------+---

## Step 6 — Reuse with different config

In [7]:
strict_config = {
    "allowed_event_types": ["purchase"],
    "min_amount": 0.0,
    "pipeline_name": "customer-events",
    "pipeline_run_id": "run_002_purchase_only",
}

strict_result = run_pipeline(bronze, strict_config)

strict_result["metrics"].show(truncate=False)
strict_result["quarantine"].select(
    "event_id",
    "event_type",
    "error_reason",
    "pipeline_run_id",
).show(truncate=False)

+---------------+-----+
|metric         |value|
+---------------+-----+
|input_rows     |6    |
|prepared_rows  |6    |
|silver_rows    |1    |
|quarantine_rows|5    |
+---------------+-----+

+--------+----------+------------------+---------------------+
|event_id|event_type|error_reason      |pipeline_run_id      |
+--------+----------+------------------+---------------------+
|e2      |purchase  |missing_user_id   |run_002_purchase_only|
|e3      |refund    |invalid_event_type|run_002_purchase_only|
|e4      |unknown   |invalid_event_type|run_002_purchase_only|
|e5      |purchase  |invalid_amount    |run_002_purchase_only|
|e6      |refund    |invalid_event_type|run_002_purchase_only|
+--------+----------+------------------+---------------------+



## Step 7 — Validate output names

In [8]:
print(result.keys())

dict_keys(['prepared', 'silver', 'quarantine', 'metrics'])


## Final result

```text
bronze
  |
  v
run_pipeline(bronze, config)
  |
  +--> prepared
  +--> silver
  +--> quarantine
  +--> metrics
```

This pattern gives a clean pipeline entry point and predictable outputs.

In [9]:
spark.stop()